# LeetCode 3590 — Kth Smallest Path XOR Sum bằng Python

- Link: https://leetcode.com/problems/kth-smallest-path-xor-sum/
- Độ khó: **Hard**
- Chủ đề: **Cây**, **XOR tiền tố trên đường đi**, **gộp nhỏ vào lớn** (small-to-large merging), **DSU on tree (Sack)**, **Fenwick tree tìm phần tử thứ k**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def kthSmallest(self, par: List[int], vals: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Tính **XOR đường đi từ gốc** cho mọi đỉnh bằng một lần duyệt BFS: `px[c] = px[cha] ^ vals[c]`.
2. Biến câu hỏi "thứ k nhỏ nhất **khác nhau** trong cây con" thành bài toán trên **tập giá trị** của cây con.
3. Nắm kỹ thuật **gộp nhỏ vào lớn**: giữ lại tập của con lớn nhất, chèn các phần tử của con nhỏ vào ⇒ mỗi phần tử bị chuyển `O(log n)` lần.
4. Biết phương án thay thế **Sack + Fenwick tree**: đếm số lần xuất hiện, đánh dấu giá trị có mặt, tìm phần tử thứ k bằng "nhảy nhị phân" trên BIT.
5. Duyệt cây **không đệ quy** (tránh tràn stack với `n = 5·10^4`).

## 2. Đề bài & ví dụ

Cây có gốc `0`, `n` đỉnh; `par[i]` là cha của `i` (`par[0] = -1`), mỗi đỉnh có giá trị `vals[i]`.

**XOR đường đi** của đỉnh `u` = XOR mọi `vals` trên đường từ gốc đến `u` (tính cả hai đầu).

Với mỗi truy vấn `[u, k]`: xét mọi đỉnh trong **cây con gốc `u`** (gồm `u` và mọi hậu duệ), lấy tập các XOR đường đi **khác nhau**, trả về giá trị **nhỏ thứ k**; nếu tập có ít hơn `k` phần tử thì trả `-1`.

**Ví dụ 1:** `par=[-1,0,0], vals=[1,1,1], queries=[[0,1],[0,2],[0,3]]` → `[0,1,-1]`. XOR đường đi: `[1,0,0]`; tập khác nhau `{0,1}`.

**Ví dụ 2:** `par=[-1,0,1], vals=[5,2,7], queries=[[0,1],[1,2],[1,3],[2,1]]` → `[0,7,-1,0]`. XOR đường đi `[5,7,0]`.

**Ràng buộc:** `1 ≤ n ≤ 5·10^4`, `0 ≤ vals[i] ≤ 10^5`, `1 ≤ len(queries) ≤ 5·10^4`, `1 ≤ k ≤ n`.

Nhận xét: `vals ≤ 10^5 < 2^17` nên mọi XOR đường đi `< 2^17 = 131072`.

In [1]:
from typing import List
from bisect import bisect_left
examples = [
    ([-1, 0, 0], [1, 1, 1], [[0, 1], [0, 2], [0, 3]], [0, 1, -1]),
    ([-1, 0, 1], [5, 2, 7], [[0, 1], [1, 2], [1, 3], [2, 1]], [0, 7, -1, 0]),
]
for par, vals, qs, exp in examples:
    print(par, vals, qs, '-> kỳ vọng', exp)

[-1, 0, 0] [1, 1, 1] [[0, 1], [0, 2], [0, 3]] -> kỳ vọng [0, 1, -1]
[-1, 0, 1] [5, 2, 7] [[0, 1], [1, 2], [1, 3], [2, 1]] -> kỳ vọng [0, 7, -1, 0]


## 3. Kiến thức nền

### 3.1. Danh sách con và thứ tự BFS

Từ mảng `par` dựng `children`. Duyệt BFS bằng cách vừa duyệt vừa nối vào danh sách `order`: cha luôn đứng trước con. Đi **ngược** `order` thì con luôn được xử lý trước cha (thứ tự "từ lá lên").

In [2]:
def build(par, vals):
    n = len(par)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    order = [0]
    px = [0] * n
    px[0] = vals[0]
    for u in order:                    # order dài dần trong lúc duyệt -> BFS
        for c in children[u]:
            px[c] = px[u] ^ vals[c]
            order.append(c)
    return children, order, px

children, order, px = build([-1, 0, 1], [5, 2, 7])
print('children =', children, '| order =', order, '| px =', px)

children = [[1], [2], []] | order = [0, 1, 2] | px = [5, 7, 0]


### 3.2. XOR

`a ^ b` là XOR từng bit. Tính chất: `x ^ x = 0`, `x ^ 0 = x`, giao hoán, kết hợp. Nhờ vậy XOR đường đi tính dồn từ cha: `px[c] = px[par[c]] ^ vals[c]`.

In [3]:
print(5 ^ 2, 5 ^ 2 ^ 7, 7 ^ 7, 9 ^ 0)
print('Giá trị XOR lớn nhất có thể (< 2^17):', (1 << 17) - 1, '>= 10^5?', (1 << 17) - 1 >= 10**5)

7 0 0 9
Giá trị XOR lớn nhất có thể (< 2^17): 131071 >= 10^5? True


### 3.3. Danh sách đã sắp + `bisect`

Một danh sách Python luôn sắp tăng có thể thay "ordered set" ở mức đơn giản:

- Kiểm tra `x` có chưa: `i = bisect_left(a, x)`, có nếu `i < len(a) and a[i] == x`.
- Chèn giữ thứ tự: `a.insert(i, x)` (dịch phần sau một ô — `memmove` rất nhanh trong C).
- Phần tử nhỏ thứ k: `a[k - 1]`.

Trên LeetCode có sẵn `sortedcontainers.SortedList` với các thao tác tương tự; notebook này chỉ dùng thư viện chuẩn.

In [4]:
def insert_unique(a, x):
    i = bisect_left(a, x)
    if i == len(a) or a[i] != x:
        a.insert(i, x)
        return True
    return False

a = []
for x in [7, 3, 7, 1, 9, 3]:
    insert_unique(a, x)
print(a, '| nhỏ thứ 2 =', a[1])

[1, 3, 7, 9] | nhỏ thứ 2 = 3


### 3.4. Fenwick tree (BIT) và tìm phần tử thứ k

BIT lưu mảng `present[v] ∈ {0,1}` (giá trị `v` có mặt hay không). Tìm `v` nhỏ nhất sao cho `present[0..v]` có tổng `≥ k` bằng **nhảy nhị phân** từ bước lớn nhất `2^17` xuống: `O(log V)`.

In [5]:
class BIT:
    def __init__(self, size_pow2):
        self.M = size_pow2
        self.t = [0] * (size_pow2 + 1)

    def add(self, v, delta):           # v là chỉ số 0-based
        i = v + 1
        while i <= self.M:
            self.t[i] += delta
            i += i & -i

    def kth(self, k):                  # giá trị 0-based nhỏ nhất có tổng tiền tố >= k
        pos, step = 0, self.M
        while step:
            if pos + step <= self.M and self.t[pos + step] < k:
                pos += step
                k -= self.t[pos]
            step >>= 1
        return pos

b = BIT(16)
for v in [3, 7, 1, 12]:
    b.add(v, 1)
print([b.kth(k) for k in range(1, 5)])    # [1, 3, 7, 12]

[1, 3, 7, 12]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bước 1: tính `px` cho mọi đỉnh. Bước 2: mỗi truy vấn chỉ hỏi về **tập** `{px[x] : x trong cây con u}`.

Tính tay ví dụ 2: cây là đường thẳng `0 — 1 — 2`, `vals = [5, 2, 7]`.

| đỉnh | px | cây con | tập px khác nhau (sắp) |
|---|---|---|---|
| 2 | 5^2^7 = 0 | {2} | [0] |
| 1 | 5^2 = 7 | {1,2} | [0, 7] |
| 0 | 5 | {0,1,2} | [0, 5, 7] |

Truy vấn `[1, 2]` → phần tử thứ 2 của `[0, 7]` là `7`; `[1, 3]` → `-1`.

Quan sát then chốt: **tập của cha = hợp các tập của con + px[cha]**. Câu hỏi là làm phép hợp sao cho nhanh.

In [6]:
children, order, px = build([-1, 0, 1], [5, 2, 7])
sets = [None] * 3
for u in reversed(order):
    s = {px[u]}
    for c in children[u]:
        s |= sets[c]
    sets[u] = s
    print('đỉnh', u, 'tập =', sorted(s))

đỉnh 2 tập = [0]
đỉnh 1 tập = [0, 7]
đỉnh 0 tập = [0, 5, 7]


## 5. Cách 1 — Brute Force: duyệt cây con cho mỗi truy vấn

Với mỗi truy vấn `[u, k]`: duyệt mọi đỉnh trong cây con `u` (DFS bằng stack), gom `px` vào `set`, sắp xếp, lấy phần tử thứ `k`.

In [7]:
def kth_brute(par, vals, queries):
    children, order, px = build(par, vals)
    res = []
    for u, k in queries:
        s, st = set(), [u]
        while st:
            x = st.pop()
            s.add(px[x])
            st.extend(children[x])
        s = sorted(s)
        res.append(s[k - 1] if k <= len(s) else -1)
    return res

for par, vals, qs, exp in examples:
    print(kth_brute(par, vals, qs), '| kỳ vọng', exp)

[0, 1, -1] | kỳ vọng [0, 1, -1]
[0, 7, -1, 0] | kỳ vọng [0, 7, -1, 0]


**Độ phức tạp:** mỗi truy vấn `O(size(u) log size(u))`, tổng `O(q · n log n)` — với `q = n = 5·10^4` lên tới cỡ `10^10` thao tác khi truy vấn dồn vào gốc.

**Nút thắt:** cùng một cây con bị duyệt lại cho nhiều truy vấn; và tập của cha được xây lại từ đầu thay vì tái sử dụng tập của con.

## 6. Ý tưởng thuật toán / bước đột phá

**Gom truy vấn theo đỉnh** (offline): xử lý cây từ lá lên, khi tập của `u` đã sẵn sàng thì trả lời mọi truy vấn hỏi `u`.

**Gộp nhỏ vào lớn:** tập của `u` = tập của **con lớn nhất** (tái sử dụng, không sao chép) + chèn các phần tử của các con còn lại + `px[u]`.

Vì sao nhanh? Mỗi khi một phần tử bị **chuyển** từ tập nhỏ sang tập lớn, tập chứa nó **ít nhất gấp đôi** kích thước. Kích thước không vượt quá `n` ⇒ mỗi phần tử bị chuyển tối đa `log2 n` lần ⇒ tổng số lần chèn `O(n log n)`.

Quy tắc:

- **Nếu** `S[c]` là tập lớn nhất trong các con **thì** dùng nó làm nền cho `S[u]`.
- **Nếu** phần tử `x` từ tập nhỏ chưa có trong nền **thì** chèn (giữ thứ tự sắp).
- Truy vấn `[u, k]`: `S[u][k-1]` nếu `k ≤ |S[u]|`, ngược lại `-1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Xây tập cho mọi đỉnh từ lá lên (sao chép đầy đủ)

Cải tiến đầu tiên: xử lý offline, mỗi đỉnh xây tập mới bằng hợp các tập con (sao chép). Trả lời truy vấn bằng danh sách đã sắp của đỉnh. Đúng, nhưng tổng kích thước các tập có thể `O(n^2)` (cây đường thẳng).

In [8]:
def kth_copy_sets(par, vals, queries):
    children, order, px = build(par, vals)
    n = len(par)
    by_node = [[] for _ in range(n)]
    for j, (u, k) in enumerate(queries):
        by_node[u].append((k, j))
    sets = [None] * n
    ans = [-1] * len(queries)
    for u in reversed(order):
        s = {px[u]}
        for c in children[u]:
            s |= sets[c]                # sao chép toàn bộ
        sets[u] = s
        if by_node[u]:
            srt = sorted(s)
            for k, j in by_node[u]:
                if k <= len(srt):
                    ans[j] = srt[k - 1]
    return ans

for par, vals, qs, exp in examples:
    print(kth_copy_sets(par, vals, qs), '| kỳ vọng', exp)

[0, 1, -1] | kỳ vọng [0, 1, -1]
[0, 7, -1, 0] | kỳ vọng [0, 7, -1, 0]


**Độ phức tạp:** `O(n^2)` thời gian và bộ nhớ trường hợp xấu.

### 7.2. Cách 3 — Gộp nhỏ vào lớn với danh sách đã sắp (lời giải chính)

Giữ cho mỗi đỉnh một **danh sách sắp tăng, không trùng**. Lấy danh sách của con lớn nhất làm nền (không sao chép), chèn phần tử của các con khác bằng `bisect`.

In [9]:
def kth_small_to_large(par, vals, queries):
    children, order, px = build(par, vals)
    n = len(par)
    by_node = [[] for _ in range(n)]
    for j, (u, k) in enumerate(queries):
        by_node[u].append((k, j))
    S = [None] * n
    ans = [-1] * len(queries)
    for u in reversed(order):
        big = None
        for c in children[u]:
            if big is None or len(S[c]) > len(big):
                big = S[c]
        if big is None:
            big = []
        for c in children[u]:
            if S[c] is not big:
                for x in S[c]:
                    insert_unique(big, x)
            S[c] = None                 # giải phóng tham chiếu
        insert_unique(big, px[u])
        S[u] = big
        for k, j in by_node[u]:
            if k <= len(big):
                ans[j] = big[k - 1]
    return ans

for par, vals, qs, exp in examples:
    print(kth_small_to_large(par, vals, qs), '| kỳ vọng', exp)

[0, 1, -1] | kỳ vọng [0, 1, -1]
[0, 7, -1, 0] | kỳ vọng [0, 7, -1, 0]


Lưu ý về cận lý thuyết: vì chỉ giữ giá trị **khác nhau**, sau khi gộp danh sách có thể không lớn gấp đôi. Để có cận chặt, hãy chọn nền là con có **cây con** (số đỉnh) lớn nhất: số phần tử bị chuyển từ con nhỏ ≤ số đỉnh của cây con nhỏ, và mỗi đỉnh chỉ nằm trong "cây con nhỏ" `O(log n)` lần trên đường lên gốc. Trong thực tế, chọn theo độ dài danh sách (như code) cũng chạy rất nhanh (xem 12.3).

**Độ phức tạp:** `O(n log n)` lần chèn; mỗi lần chèn là `bisect` `O(log n)` + `memmove` (rất nhanh nhưng về lý thuyết `O(n)`). Space `O(n)`.

### 7.3. Cách 4 — Sack (DSU on tree) + Fenwick tree

Một cách có cận lý thuyết chặt `O(n log n log V)`:

1. Tính kích thước cây con, **con nặng** (heavy child), và thứ tự Euler sao cho cây con là một đoạn liên tục `[tin[u], tin[u] + size[u])`.
2. Duyệt `dfs(u, keep)`: xử lý các con nhẹ với `keep = False` (xong thì xóa sạch), con nặng với `keep = True` (giữ lại), rồi thêm mọi đỉnh của các con nhẹ và `u` vào cấu trúc đếm, trả lời truy vấn của `u`; nếu `keep = False` thì xóa cả cây con `u`.
3. Cấu trúc đếm: `cnt[v]` số đỉnh có `px = v`; khi `cnt[v]` đổi giữa 0 và 1 thì cập nhật BIT. Truy vấn thứ k = `bit.kth(k)`.

In [10]:
def kth_sack(par, vals, queries):
    children, order, px = build(par, vals)
    n = len(par)
    size = [1] * n
    for u in reversed(order):
        if u:
            size[par[u]] += size[u]
    heavy = [-1] * n
    for u in range(n):
        if children[u]:
            heavy[u] = max(children[u], key=size.__getitem__)
    tin, euler = [0] * n, [0] * n
    t, st = 0, [0]
    while st:                            # DFS tiền thứ tự -> cây con liên tục
        u = st.pop()
        tin[u], euler[t] = t, u
        t += 1
        st.extend(c for c in children[u] if c != heavy[u])
        if heavy[u] != -1:
            st.append(heavy[u])
    by_node = [[] for _ in range(n)]
    for j, (u, k) in enumerate(queries):
        by_node[u].append((k, j))

    M = 1 << 17
    bit = BIT(M)
    cnt = [0] * M
    distinct = [0]

    def add(v):
        cnt[v] += 1
        if cnt[v] == 1:
            distinct[0] += 1
            bit.add(v, 1)

    def remove(v):
        cnt[v] -= 1
        if cnt[v] == 0:
            distinct[0] -= 1
            bit.add(v, -1)

    ans = [-1] * len(queries)
    stack = [(0, True, 0)]               # (đỉnh, keep, pha)
    while stack:
        u, keep, phase = stack.pop()
        if phase == 0:
            stack.append((u, keep, 1))
            if heavy[u] != -1:
                stack.append((heavy[u], True, 0))   # xử lý SAU các con nhẹ
            for c in children[u]:
                if c != heavy[u]:
                    stack.append((c, False, 0))
        else:
            for c in children[u]:
                if c != heavy[u]:
                    for i in range(tin[c], tin[c] + size[c]):
                        add(px[euler[i]])
            add(px[u])
            for k, j in by_node[u]:
                if k <= distinct[0]:
                    ans[j] = bit.kth(k)
            if not keep:
                for i in range(tin[u], tin[u] + size[u]):
                    remove(px[euler[i]])
    return ans

for par, vals, qs, exp in examples:
    print(kth_sack(par, vals, qs), '| kỳ vọng', exp)

[0, 1, -1] | kỳ vọng [0, 1, -1]
[0, 7, -1, 0] | kỳ vọng [0, 7, -1, 0]


Thứ tự trên stack: con nặng được đẩy **trước** các con nhẹ nên được lấy ra **sau cùng** — đúng yêu cầu "xử lý con nhẹ (và dọn sạch) trước, con nặng sau để giữ lại".

**Độ phức tạp:** mỗi đỉnh bị thêm/xóa `O(log n)` lần (mỗi lần nó nằm trong một cây con nhẹ), mỗi lần `O(log V)` trên BIT ⇒ `O(n log n log V + q log V)`.

## 8. TRACE từng bước

Trace cách 3 trên cây 6 đỉnh:

```
        0 (vals 3)
       / \
  (1) 1   2 (6)
     / \    \
(1) 3  4(2)  5 (5)
```

`px = [3, 2, 5, 3, 0, 0]`. Từ lá lên: các lá có danh sách 1 phần tử; đỉnh 1 lấy danh sách của con 3 làm nền rồi chèn phần tử của con 4; ...

In [11]:
def trace_small_to_large(par, vals, queries):
    children, order, px = build(par, vals)
    print('px =', px)
    n = len(par)
    by_node = [[] for _ in range(n)]
    for j, (u, k) in enumerate(queries):
        by_node[u].append((k, j))
    S = [None] * n
    ans = [-1] * len(queries)
    moved = 0
    for u in reversed(order):
        big, big_child = [], None
        for c in children[u]:
            if big_child is None or len(S[c]) > len(S[big_child]):
                big_child = c
        if big_child is not None:
            big = S[big_child]
        notes = []
        for c in children[u]:
            if c != big_child:
                for x in S[c]:
                    moved += 1
                    notes.append(f"chèn {x}{'' if insert_unique(big, x) else ' (trùng)'}")
        ins = insert_unique(big, px[u])
        S[u] = big
        print(f"đỉnh {u}: nền = con {big_child}, {', '.join(notes) or 'không chèn từ con nhỏ'}; "
              f"thêm px[{u}]={px[u]}{'' if ins else ' (trùng)'} -> {big}")
        for k, j in by_node[u]:
            ans[j] = big[k - 1] if k <= len(big) else -1
            print(f"    truy vấn [{u}, {k}] -> {ans[j]}")
    print('Tổng số phần tử bị chuyển từ tập nhỏ:', moved)
    return ans

par6 = [-1, 0, 0, 1, 1, 2]
vals6 = [3, 1, 6, 1, 2, 5]
q6 = [[0, 1], [0, 3], [0, 4], [1, 2], [2, 2], [5, 1]]
trace_small_to_large(par6, vals6, q6)

px = [3, 2, 5, 3, 0, 0]
đỉnh 5: nền = con None, không chèn từ con nhỏ; thêm px[5]=0 -> [0]
    truy vấn [5, 1] -> 0
đỉnh 4: nền = con None, không chèn từ con nhỏ; thêm px[4]=0 -> [0]
đỉnh 3: nền = con None, không chèn từ con nhỏ; thêm px[3]=3 -> [3]
đỉnh 2: nền = con 5, không chèn từ con nhỏ; thêm px[2]=5 -> [0, 5]
    truy vấn [2, 2] -> 5
đỉnh 1: nền = con 3, chèn 0; thêm px[1]=2 -> [0, 2, 3]
    truy vấn [1, 2] -> 2
đỉnh 0: nền = con 1, chèn 0 (trùng), chèn 5; thêm px[0]=3 (trùng) -> [0, 2, 3, 5]
    truy vấn [0, 1] -> 0
    truy vấn [0, 3] -> 3
    truy vấn [0, 4] -> 5
Tổng số phần tử bị chuyển từ tập nhỏ: 3


[0, 3, 5, 2, 5, 0]

Đối chiếu nhanh: tập của gốc là `{0, 2, 3, 5}`, nên `[0,3] → 3`, `[0,4] → 5`. Đỉnh 1 có `{0, 2, 3}` ⇒ `[1,2] → 2`.

In [12]:
print('vét cạn:', kth_brute(par6, vals6, q6))

vét cạn: [0, 3, 5, 2, 5, 0]


## 9. Code nộp LeetCode

Lời giải chính (gộp nhỏ vào lớn, chỉ dùng thư viện chuẩn). Duyệt không đệ quy.

In [13]:
class Solution:
    def kthSmallest(self, par: List[int], vals: List[int], queries: List[List[int]]) -> List[int]:
        n = len(par)
        children = [[] for _ in range(n)]
        for i in range(1, n):
            children[par[i]].append(i)
        order = [0]
        px = [0] * n
        px[0] = vals[0]
        for u in order:
            for c in children[u]:
                px[c] = px[u] ^ vals[c]
                order.append(c)

        by_node = [[] for _ in range(n)]
        for j, (u, k) in enumerate(queries):
            by_node[u].append((k, j))

        S = [None] * n
        ans = [-1] * len(queries)
        for u in reversed(order):
            big = None
            for c in children[u]:
                if big is None or len(S[c]) > len(big):
                    big = S[c]
            if big is None:
                big = []
            for c in children[u]:
                small = S[c]
                if small is not big:
                    for x in small:
                        i = bisect_left(big, x)
                        if i == len(big) or big[i] != x:
                            big.insert(i, x)
                S[c] = None
            x = px[u]
            i = bisect_left(big, x)
            if i == len(big) or big[i] != x:
                big.insert(i, x)
            S[u] = big
            for k, j in by_node[u]:
                if k <= len(big):
                    ans[j] = big[k - 1]
        return ans

for par, vals, qs, exp in examples:
    got = Solution().kthSmallest(par, vals, qs)
    print(got, 'OK' if got == exp else 'SAI')

[0, 1, -1] OK
[0, 7, -1, 0] OK


## 10. Vì sao thuật toán đúng

1. **XOR đường đi:** `px[c] = px[par[c]] ^ vals[c]` theo định nghĩa (đường từ gốc tới `c` = đường tới cha + `c`). BFS đảm bảo cha được tính trước con.
2. **Bất biến của cách 3:** sau khi xử lý `u`, `S[u]` là danh sách **sắp tăng, không trùng** đúng bằng `{px[x] : x ∈ subtree(u)}`. Chứng minh quy nạp từ lá lên: cây con `u` = `{u}` ∪ hợp các cây con của con; ta lấy hợp đúng tập đó, chèn có kiểm tra trùng và giữ thứ tự bằng `bisect`.
3. **Không phá dữ liệu còn cần:** `S[c]` của con chỉ được dùng khi xử lý cha trực tiếp, nên tái sử dụng danh sách của con lớn nhất làm `S[u]` không ảnh hưởng truy vấn nào khác (truy vấn của `c` đã được trả lời lúc xử lý `c`).
4. **Trả lời truy vấn:** phần tử nhỏ thứ `k` của danh sách sắp tăng không trùng là `S[u][k-1]`; nếu ít hơn `k` phần tử thì `-1`.
5. **Cách 4:** tại thời điểm trả lời truy vấn của `u`, cấu trúc đếm chứa chính xác các đỉnh của cây con `u` (con nặng được giữ lại, con nhẹ thêm lại, các cây con khác đã bị xóa khi `keep = False`). `cnt[v] > 0` ⇔ `v` có mặt; BIT đếm số giá trị có mặt ≤ `v`, nên `kth` trả về giá trị nhỏ thứ `k`.

## 11. Phân tích Time / Space Complexity

Gọi `n` số đỉnh, `q` số truy vấn, `V = 2^17`.

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt cây con mỗi truy vấn | `O(q · n log n)` | `O(n)` |
| 2. Sao chép tập từ lá lên | `O(n^2)` | `O(n^2)` |
| 3. Gộp nhỏ vào lớn (list + bisect) | `O(n log^2 n)` so sánh + chi phí `memmove` | `O(n)` |
| 4. Sack + BIT | `O(n log n log V + q log V)` | `O(n + V)` |

Thực tế với `n = q = 5·10^4`, cả cách 3 và 4 chạy dưới ~1 giây trong Python (xem 12.3).

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'copy_sets': kth_copy_sets,
    'small_to_large': kth_small_to_large,
    'sack': kth_sack,
    'submit': lambda p, v, q: Solution().kthSmallest(p, v, q),
}
edge = [
    ([-1], [0], [[0, 1]], [0]),                      # một đỉnh, vals = 0
    ([-1], [100000], [[0, 1], [0, 1]], [100000, 100000]),
    ([-1, 0, 1, 2], [4, 4, 4, 4], [[0, 1], [0, 2], [0, 3], [3, 1]], [0, 4, -1, 0]),  # XOR xen kẽ 4,0,4,0
    ([-1, 0, 0, 0], [1, 2, 3, 0], [[0, 3], [0, 4], [1, 1]], [3, -1, 3]),   # px = [1, 3, 2, 1]
]
for par, vals, qs, exp in examples + edge:
    for name, f in impls.items():
        got = f(par, vals, qs)
        assert got == exp, (name, par, vals, qs, got, exp)
    assert kth_brute(par, vals, qs) == exp
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [15]:
import random
random.seed(3590)
cnt = 0
for _ in range(500):
    n = random.randint(1, 15)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [random.randint(0, 15) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, n)] for _ in range(random.randint(1, 12))]
    exp = kth_brute(par, vals, qs)
    for name, f in impls.items():
        assert f(par, vals, qs) == exp, (name, par, vals, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cây ngẫu nhiên.')

Khớp vét cạn trên 500 cây ngẫu nhiên.


### 12.3. Ca lớn (`n = q = 5·10^4`): cây ngẫu nhiên, đường thẳng, cây "chổi"

In [16]:
import time
random.seed(7)
n = 5 * 10**4
shapes = {
    'ngẫu nhiên': [-1] + [random.randrange(i) for i in range(1, n)],
    'đường thẳng': [-1] + list(range(n - 1)),
    'chổi (sao)': [-1] + [0] * (n - 1),
}
for name, par in shapes.items():
    vals = [random.randint(0, 10**5) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, n)] for _ in range(n)]
    t = time.time(); a = Solution().kthSmallest(par, vals, qs); t1 = time.time() - t
    t = time.time(); b = kth_sack(par, vals, qs); t2 = time.time() - t
    assert a == b
    print(f"{name:>12}: small-to-large {t1:.2f}s | sack+BIT {t2:.2f}s")

  ngẫu nhiên: small-to-large 0.30s | sack+BIT 0.70s


 đường thẳng: small-to-large 0.23s | sack+BIT 0.33s


  chổi (sao): small-to-large 0.24s | sack+BIT 0.31s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm cả giá trị trùng (quên "distinct")

In [17]:
def wrong_not_distinct(par, vals, queries):
    children, order, px = build(par, vals)
    res = []
    for u, k in queries:
        lst, st = [], [u]
        while st:
            x = st.pop()
            lst.append(px[x])          # SAI: giữ cả giá trị trùng
            st.extend(children[x])
        lst.sort()
        res.append(lst[k - 1] if k <= len(lst) else -1)
    return res

par, vals, qs, exp = examples[0]
print('sai :', wrong_not_distinct(par, vals, qs))   # [0, 0, 1]
print('đúng:', Solution().kthSmallest(par, vals, qs))

sai : [0, 0, 1]
đúng: [0, 1, -1]


**Vì sao sai:** đề hỏi giá trị nhỏ thứ k trong các XOR **khác nhau**; hai đỉnh cùng XOR 0 chỉ tính một lần.

### Lỗi 2 — Dùng `vals[u]` thay vì XOR **cả đường đi**

In [18]:
def wrong_use_vals(par, vals, queries):
    res = []
    for u, k in queries:
        s = sorted({vals[x] for x in collect(par, u)})   # SAI: dùng vals thay vì px
        res.append(s[k - 1] if k <= len(s) else -1)
    return res

def collect(par, u):
    n = len(par)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    out, st = [], [u]
    while st:
        x = st.pop()
        out.append(x)
        st.extend(children[x])
    return out

par, vals, qs, exp = examples[1]
print('sai :', wrong_use_vals(par, vals, qs))
print('đúng:', Solution().kthSmallest(par, vals, qs))

sai : [2, 7, -1, 7]
đúng: [0, 7, -1, 0]


**Vì sao sai:** giá trị cần xét là XOR từ **gốc của cả cây** (đỉnh 0) tới đỉnh, không phải giá trị riêng của đỉnh, cũng không phải XOR từ `u`.

### Lỗi 3 — Gộp con lớn vào con nhỏ (sai hướng) — đúng nhưng chậm

In [19]:
def merge_count(par, big_into_small):
    """Đếm số lần chèn khi gộp; big_into_small=True là hướng SAI."""
    children, order, _ = build(par, [0] * len(par))
    sz = [1] * len(par)
    total = 0
    for u in reversed(order):
        if not children[u]:
            continue
        cs = sorted(children[u], key=lambda c: sz[c])
        base = cs[0] if big_into_small else cs[-1]
        for c in children[u]:
            if c != base:
                total += sz[c]
        total += 1
        sz[u] = 1 + sum(sz[c] for c in children[u])
    return total

# cây "sâu đậm": mỗi đỉnh trên xương sống có thêm một lá
m = 3000
par_cat = [-1]
for i in range(1, m):
    par_cat.append(i - 1)          # xương sống
for i in range(m):
    par_cat.append(i)              # mỗi đỉnh xương sống thêm 1 lá
print('đúng hướng (nhỏ vào lớn):', merge_count(par_cat, False))
print('sai hướng (lớn vào nhỏ) :', merge_count(par_cat, True))

đúng hướng (nhỏ vào lớn): 5999
sai hướng (lớn vào nhỏ) : 9000000


**Vì sao sai:** gộp lớn vào nhỏ làm mất cận `log n`; trên cây "xương sống + lá", số lần chèn tăng từ `O(n)` lên `O(n^2)` ⇒ TLE (kết quả vẫn đúng nhưng quá chậm).

### Lỗi 4 — Dùng đệ quy sâu với cây đường thẳng

In [20]:
import sys
def depth_recursive(children, u):
    return 1 + max((depth_recursive(children, c) for c in children[u]), default=0)

chain = [-1] + list(range(4999))
children_chain, _, _ = build(chain, [0] * 5000)
try:
    depth_recursive(children_chain, 0)
    print('chạy được (giới hạn đệ quy đủ lớn)')
except RecursionError as e:
    print('RecursionError:', str(e)[:60], '-> nên duyệt bằng stack/BFS')
print('giới hạn đệ quy hiện tại:', sys.getrecursionlimit())

RecursionError: maximum recursion depth exceeded -> nên duyệt bằng stack/BFS
giới hạn đệ quy hiện tại: 1000


**Vì sao sai:** giới hạn đệ quy mặc định của Python (~1000) nhỏ hơn độ sâu cây (tới `5·10^4`). Dùng thứ tự BFS + đi ngược như lời giải chính.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Giá trị trên **đường từ gốc**" với phép kết hợp có nghịch đảo (XOR, cộng) ⇒ **tiền tố trên cây**: `px[c] = px[cha] op vals[c]`.
- "Truy vấn về **tập giá trị trong cây con**" ⇒ offline từ lá lên + **gộp nhỏ vào lớn**, hoặc **Euler tour** + **Sack**.
- "Thứ k nhỏ nhất" ⇒ danh sách sắp + `bisect`, `SortedList`, hoặc BIT + nhảy nhị phân (khi miền giá trị nhỏ ~`2^17`).
- "Khác nhau" ⇒ đếm `cnt[v]` và chỉ cập nhật BIT khi `cnt` đổi giữa 0 và 1.
- `n` tới `10^5` trên cây ⇒ **không đệ quy** trong Python.

## 15. Biến thể của bài

1. Hỏi **số lượng** XOR khác nhau trong cây con (LC 1981-style "distinct in subtree") ⇒ chỉ cần `len(S[u])`.
2. Hỏi thứ k **lớn nhất** ⇒ `S[u][len - k]`.
3. Thay "cây con" bằng "đường đi `u → v`" ⇒ cần cây phân đoạn bền vững (persistent) + LCA.
4. Có cập nhật `vals` ⇒ XOR của cả cây con của đỉnh bị đổi thay đổi, cần Euler tour + cấu trúc phức tạp hơn.
5. Cùng kỹ thuật: LC 1916, CF 600E *Lomsat gelral* (Sack), LC 2003 *Smallest Missing Genetic Value in Each Subtree*.

In [21]:
# Biến thể 1 & 2: số lượng khác nhau và thứ k lớn nhất
def subtree_sets(par, vals):
    children, order, px = build(par, vals)
    S = [None] * len(par)
    for u in reversed(order):
        big = max((S[c] for c in children[u]), key=len, default=None)
        big = big if big is not None else []
        for c in children[u]:
            if S[c] is not big:
                for x in S[c]:
                    insert_unique(big, x)
        insert_unique(big, px[u])
        S[u] = big
    return S                     # lưu ý: các danh sách bị chia sẻ, chỉ đúng tại gốc

S = subtree_sets(par6, vals6)
print('số XOR khác nhau tại gốc:', len(S[0]), '| lớn nhất thứ 2:', S[0][-2])

số XOR khác nhau tại gốc: 4 | lớn nhất thứ 2: 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Duyệt cây con mỗi truy vấn | `O(q n log n)` | `O(n)` | Oracle kiểm thử |
| 2 | Sao chép tập từ lá lên | `O(n^2)` | `O(n^2)` | Hiểu ý tưởng offline |
| 3 | Gộp nhỏ vào lớn (list + bisect) | ~`O(n log^2 n)` | `O(n)` | **Nộp bài** (hoặc `SortedList`) |
| 4 | Sack + BIT | `O(n log n log V)` | `O(n + V)` | Khi cần cận chặt, miền giá trị nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `par=[-1,0,0,1], vals=[2,3,2,3]`, truy vấn `[[0,1],[0,2],[1,3]]` = ?

**Bài 2 (tính tay):** Với cây ở mục 8, truy vấn `[2, 1]` và `[2, 3]` trả về gì?

**Bài 3 (điền TODO):** viết `prefix_xor(par, vals)` trả về `px` (không đệ quy).

**Bài 4 (tự viết lại):** viết `count_distinct_subtree(par, vals)` trả về số XOR khác nhau của **mọi** cây con (lưu ý danh sách bị chia sẻ — phải lưu độ dài ngay khi xử lý xong `u`).

**Bài 5 (mini challenge):** trả lời truy vấn "thứ k **lớn nhất**" bằng cách 4 (Sack + BIT).

In [22]:
def prefix_xor_todo(par, vals):
    # TODO: dựng children, BFS từ gốc, px[c] = px[u] ^ vals[c]
    raise NotImplementedError

def count_distinct_subtree_todo(par, vals):
    # TODO: gộp nhỏ vào lớn, ghi len(S[u]) ngay sau khi xử lý u
    raise NotImplementedError

def kth_largest_sack_todo(par, vals, queries):
    # TODO: như kth_sack nhưng đổi k thành distinct - k + 1
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `px = [2, 1, 0, 2]`. (2) Cây con 0: `{0,1,2}`; cây con 1: `{1,2}`. (3) `[0, 1, -1]`.

**Bài 2.** (1) Cây con 2 = `{2, 5}`. (2) `px[2] = 5`, `px[5] = 0`. (3) `[2,1] → 0`, `[2,3] → -1`.

**Bài 3.** (1) Như hàm `build`. (2) Duyệt `order` trong khi nối thêm. (3) Kiểm tra với vét cạn đi ngược lên cha.

**Bài 4.** (1) Danh sách của con lớn nhất bị "mượn" cho cha. (2) Ghi `res[u] = len(big)` ngay sau khi chèn `px[u]`. (3) So với vét cạn.

**Bài 5.** (1) Phần tử lớn thứ k = nhỏ thứ `distinct - k + 1`. (2) Kiểm tra `k ≤ distinct`. (3) So với vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
a1 = Solution().kthSmallest([-1, 0, 0, 1], [2, 3, 2, 3], [[0, 1], [0, 2], [1, 3]])
a2 = Solution().kthSmallest(par6, vals6, [[2, 1], [2, 3]])
assert a1 == kth_brute([-1, 0, 0, 1], [2, 3, 2, 3], [[0, 1], [0, 2], [1, 3]]) == [0, 1, -1]
assert a2 == [0, -1]
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: [0, 1, -1] | Bài 2: [0, -1]


In [24]:
# Bài 3
def prefix_xor(par, vals):
    n = len(par)
    children = [[] for _ in range(n)]
    for i in range(1, n):
        children[par[i]].append(i)
    px = [0] * n
    px[0] = vals[0]
    order = [0]
    for u in order:
        for c in children[u]:
            px[c] = px[u] ^ vals[c]
            order.append(c)
    return px

def prefix_xor_slow(par, vals):
    res = []
    for u in range(len(par)):
        x, v = 0, u
        while v != -1:
            x ^= vals[v]
            v = par[v]
        res.append(x)
    return res

random.seed(31)
for _ in range(200):
    n = random.randint(1, 20)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [random.randint(0, 100) for _ in range(n)]
    assert prefix_xor(par, vals) == prefix_xor_slow(par, vals)
print('Bài 3: prefix_xor khớp cách đi ngược lên gốc')

Bài 3: prefix_xor khớp cách đi ngược lên gốc


In [25]:
# Bài 4
def count_distinct_subtree(par, vals):
    children, order, px = build(par, vals)
    S = [None] * len(par)
    res = [0] * len(par)
    for u in reversed(order):
        big = None
        for c in children[u]:
            if big is None or len(S[c]) > len(big):
                big = S[c]
        big = [] if big is None else big
        for c in children[u]:
            if S[c] is not big:
                for x in S[c]:
                    insert_unique(big, x)
        insert_unique(big, px[u])
        S[u] = big
        res[u] = len(big)
    return res

random.seed(32)
for _ in range(200):
    n = random.randint(1, 15)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [random.randint(0, 7) for _ in range(n)]
    _, _, px = build(par, vals)
    exp = [len({px[x] for x in collect(par, u)}) for u in range(n)]
    assert count_distinct_subtree(par, vals) == exp
print('Bài 4: ví dụ', count_distinct_subtree(par6, vals6))

Bài 4: ví dụ [4, 3, 2, 1, 1, 1]


In [26]:
# Bài 5: thứ k lớn nhất bằng Sack + BIT (tái dùng kth_sack bằng cách đổi truy vấn)
def kth_largest(par, vals, queries):
    # đếm số distinct mỗi cây con, rồi đổi k -> distinct - k + 1
    dc = count_distinct_subtree(par, vals)
    conv = [[u, dc[u] - k + 1 if k <= dc[u] else 10**9] for u, k in queries]
    res = kth_sack(par, vals, [[u, k] for u, k in conv])
    return [r if k <= dc[u] else -1 for r, (u, k) in zip(res, queries)]

def kth_largest_brute(par, vals, queries):
    _, _, px = build(par, vals)
    out = []
    for u, k in queries:
        s = sorted({px[x] for x in collect(par, u)}, reverse=True)
        out.append(s[k - 1] if k <= len(s) else -1)
    return out

random.seed(33)
for _ in range(200):
    n = random.randint(1, 12)
    par = [-1] + [random.randrange(i) for i in range(1, n)]
    vals = [random.randint(0, 15) for _ in range(n)]
    qs = [[random.randrange(n), random.randint(1, n)] for _ in range(6)]
    assert kth_largest(par, vals, qs) == kth_largest_brute(par, vals, qs)
print('Bài 5: ví dụ', kth_largest(par6, vals6, [[0, 1], [0, 2], [1, 3]]))

Bài 5: ví dụ [5, 3, 0]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
px[0] = vals[0]; BFS: px[c] = px[u] ^ vals[c]; order = thứ tự BFS
gom truy vấn theo đỉnh
for u in reversed(order):
    big = danh sách của con lớn nhất (hoặc [])
    chèn (bisect, bỏ trùng) mọi phần tử của các con khác vào big
    chèn px[u]; S[u] = big
    trả lời [u, k]: big[k-1] nếu k <= len(big) else -1
```

**Checklist:**

- [ ] XOR tính từ **gốc 0** (không phải từ `u`)?
- [ ] Chỉ đếm giá trị **khác nhau**?
- [ ] Gộp **nhỏ vào lớn**, không sao chép danh sách lớn?
- [ ] Trả lời truy vấn **ngay** khi xử lý xong `u` (trước khi danh sách bị cha mượn)?
- [ ] Không đệ quy (cây đường thẳng `5·10^4` đỉnh)?
- [ ] Trả `-1` khi `k` vượt số giá trị khác nhau?